# 4.9 环境和分布偏移

训练集表现优秀并不保证部署成功。监督学习通常隐含假设训练数据与未来数据来自相同分布，但现实中的用户、设备、时间、政策和采集流程都会改变。模型不仅要拟合数据，还必须明确它将在什么环境中使用。

原教材：[4.9 环境和分布偏移](https://zh.d2l.ai/chapter_multilayer-perceptrons/environment.html)

## 学习目标

1. 区分协变量偏移、标签偏移和概念偏移；
2. 理解经验风险最小化为何依赖同分布假设；
3. 掌握协变量偏移下的重要性加权思想；
4. 认识部署、反馈回路、公平性与因果关系带来的风险。

## 4.9.1 风险与分布

训练通常最小化经验风险

$$\hat R_{train}(f)=\frac1n\sum_{i=1}^{n}l(f(\mathbf x_i),y_i),$$

而真正关心的是部署分布上的期望风险

$$R_{test}(f)=\mathbb E_{(\mathbf x,y)\sim p_{test}}[l(f(\mathbf x),y)].$$

只有当训练样本能代表部署环境时，训练风险才是测试风险的可靠估计。偏移问题的关键不是简单地问“数据变了吗”，而是判断联合分布 $p(\mathbf x,y)$ 的哪一部分发生了变化。

## 4.9.2 三类分布偏移

| 类型 | 保持不变 | 发生变化 | 例子 |
|---|---|---|---|
| 协变量偏移 | $p(y\mid x)$ | $p(x)$ | 训练图片来自专业相机，部署图片来自手机 |
| 标签偏移 | $p(x\mid y)$ | $p(y)$ | 疾病检测方法不变，但不同地区患病率不同 |
| 概念偏移 | 无法假设 $p(y\mid x)$ 不变 | 标签规则或机制改变 | 欺诈者适应检测系统，旧模式不再代表欺诈 |

现实偏移可能同时包含多种类型。分类的价值在于决定哪些校正方法有理论依据，而不是给问题贴标签后停止分析。

## 4.9.3 协变量偏移与重要性加权

若假设 $p_{train}(y\mid x)=p_{test}(y\mid x)$，则可以重写测试风险：

$$R_{test}(f)=\int l(f(x),y)p_{test}(x,y)\,dxdy=\mathbb E_{train}\left[\frac{p_{test}(x)}{p_{train}(x)}l(f(x),y)\right].$$

因此训练样本应乘重要性权重 $\beta(x)=p_{test}(x)/p_{train}(x)$。测试环境中更常见的区域获得更大权重。若两个分布重叠很少，权重可能极大，估计方差也会很高；没有训练支持的区域不能靠重新加权凭空创造知识。

In [ ]:
from pathlib import Path  # 统一管理本地与 Google Drive 中的图片路径

import matplotlib.pyplot as plt  # 绘制训练分布、测试分布与决策边界
import torch  # 提供张量计算、自动微分与随机采样
from torch import nn  # 提供线性分类模型和二元交叉熵损失

try:  # 尝试判断当前环境是否为 Google Colab
    from google.colab import drive  # 导入 Colab 的云盘挂载工具
    drive.mount('/content/drive')  # 挂载用户的“我的云端硬盘”
    project_root = Path('/content/drive/MyDrive/d2l_learning')  # 指向云盘中的项目根目录
except ImportError:  # 本地环境没有 google.colab 时进入此分支
    project_root = Path.cwd().parent if Path.cwd().name == 'Chapter_4' else Path.cwd()  # 推断本地项目根目录

figure_dir = project_root / 'Chapter_4' / 'images'  # 设置第 4 章图片保存目录
figure_dir.mkdir(parents=True, exist_ok=True)  # 创建图片目录且允许目录已存在
torch.manual_seed(42)  # 固定 CPU 随机种子以复现实验结果
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))  # 自动选择 Colab GPU 或本地 CPU
print('计算设备:', device)  # 输出当前实际使用的计算设备

In [ ]:
def normal_pdf(x, mean, std):  # 定义一维正态分布概率密度函数
    coefficient = 1.0 / (std * (2.0 * torch.pi) ** 0.5)  # 计算正态密度的归一化系数
    return coefficient * torch.exp(-0.5 * ((x - mean) / std).pow(2))  # 返回每个位置的正态密度

num_train, num_test = 1000, 1000  # 设置训练样本数与测试样本数
train_x = torch.randn(num_train, 1, device=device) * 1.0 - 1.0  # 从均值 -1 的训练输入分布采样
test_x = torch.randn(num_test, 1, device=device) * 1.0 + 1.0  # 从均值 1 的测试输入分布采样
train_y = (train_x + 0.35 * torch.randn_like(train_x) > 0).float()  # 使用同一条件机制生成训练标签
test_y = (test_x + 0.35 * torch.randn_like(test_x) > 0).float()  # 使用同一条件机制生成测试标签
importance = normal_pdf(train_x, 1.0, 1.0) / normal_pdf(train_x, -1.0, 1.0)  # 计算测试密度与训练密度之比
importance = importance.clamp(max=20.0)  # 截断极端权重以控制有限样本估计方差
print('重要性权重范围:', importance.min().item(), importance.max().item())  # 输出权重范围以检查稳定性

In [ ]:
def train_classifier(sample_weights=None, num_epochs=300):  # 定义可选择重要性加权的逻辑回归训练函数
    model = nn.Linear(1, 1).to(device)  # 创建并移动一维线性分类器到计算设备
    nn.init.zeros_(model.weight)  # 将分类器权重初始化为零
    nn.init.zeros_(model.bias)  # 将分类器偏置初始化为零
    loss_fn = nn.BCEWithLogitsLoss(reduction='none')  # 创建返回逐样本损失的稳定二元交叉熵
    optimizer = torch.optim.Adam(model.parameters(), lr=0.03)  # 创建 Adam 优化器
    for epoch in range(num_epochs):  # 重复执行指定轮数的全批量训练
        optimizer.zero_grad()  # 清除上一轮累积在参数上的梯度
        per_example_loss = loss_fn(model(train_x), train_y)  # 计算每个训练样本的独立损失
        loss = per_example_loss.mean() if sample_weights is None else (sample_weights * per_example_loss).sum() / sample_weights.sum()  # 选择普通或加权经验风险
        loss.backward()  # 反向传播计算分类器参数梯度
        optimizer.step()  # 根据梯度更新权重和偏置
    return model  # 返回训练完成的分类器

plain_model = train_classifier()  # 使用普通经验风险训练基线模型
weighted_model = train_classifier(importance)  # 使用协变量偏移重要性权重训练模型

def accuracy(model, X, y):  # 定义二分类准确率计算函数
    with torch.no_grad():  # 关闭梯度记录以节省评估资源
        predictions = (model(X) >= 0).float()  # 以 logit 是否大于等于零作为类别判定
        return (predictions == y).float().mean().item()  # 返回预测正确比例

print('普通模型测试准确率:', accuracy(plain_model, test_x, test_y))  # 输出未加权模型在偏移测试集上的准确率
print('加权模型测试准确率:', accuracy(weighted_model, test_x, test_y))  # 输出重要性加权模型的测试准确率

In [ ]:
grid = torch.linspace(-4, 4, 400, device=device).reshape(-1, 1)  # 创建用于绘制密度曲线的连续横坐标
fig, axis = plt.subplots(figsize=(8, 4))  # 创建训练与测试分布比较图画布
axis.plot(grid.cpu(), normal_pdf(grid, -1.0, 1.0).cpu(), label='train p(x)')  # 绘制训练输入概率密度
axis.plot(grid.cpu(), normal_pdf(grid, 1.0, 1.0).cpu(), label='test p(x)')  # 绘制测试输入概率密度
axis.set_xlabel('x')  # 设置横轴为输入变量 x
axis.set_ylabel('density')  # 设置纵轴为概率密度
axis.grid(alpha=0.3)  # 添加半透明网格辅助读数
axis.legend()  # 显示训练分布和测试分布图例
plt.tight_layout()  # 自动调整画布边距
figure_path = figure_dir / '4.9_covariate_shift.png'  # 生成图片的完整保存路径
fig.savefig(figure_path, dpi=160, bbox_inches='tight')  # 保存高清图片并裁掉多余白边
print(f'图片已保存到：{figure_path}')  # 输出图片保存位置
plt.show()  # 在 Notebook 中显示分布偏移图

### 如何解释加权实验结果

重要性加权校正的是**目标风险的估计方式**，并不保证有限样本下的准确率一定提高。本例的逻辑回归已经能较好表示条件关系 $p(y\mid x)$，因此普通模型与加权模型可能表现接近，加权模型甚至会因权重方差而略差。若模型有偏、训练样本有限或测试分布更强调某些区域，加权才更可能改变最优参数。实际使用时应同时报告权重分布、有效样本量和目标环境验证指标。

## 4.9.4 标签偏移与概念偏移

标签偏移假设类别内部的特征分布 $p(x\mid y)$ 不变，只是类别先验 $p(y)$ 改变。此时可以估计部署环境的类别比例，并校正预测概率或按类别重新加权。该假设在“患病率改变但疾病表现方式稳定”等场景可能近似成立。

概念偏移意味着 $p(y\mid x)$ 改变。例如价格、语言、欺诈策略和用户偏好随时间变化。仅调整样本权重通常无法修复概念偏移，需要收集新标签、持续监控并重新训练。缓慢变化称为漂移，突发政策或产品变化可能导致跳变。

## 4.9.5 机器学习部署中的问题

### 非平稳环境和反馈回路

模型的预测可能改变未来数据。推荐系统决定用户看到什么，信贷模型影响谁能获得贷款，内容审核改变发布行为。部署后观测到的数据不再是被动样本，而是模型与环境共同产生的结果。只在历史日志上离线评估可能忽略这种策略反馈。

### 虚假相关与因果关系

训练分布中的相关特征可能只是环境线索。例如牛常出现在草地、医疗标签可能泄漏医院流程。环境变化时，这些捷径会失效。预测任务不总要求完整因果模型，但若模型用于干预或跨环境部署，就必须问清哪些关系在机制变化后仍保持稳定。

### 公平性、安全与监控

总体准确率可能掩盖子群体上的严重退化。实践中应按时间、地区、设备和关键人群分层监控输入分布、缺失率、输出置信度、延迟标签下的性能和业务后果，并为回滚、人工复核与重新训练建立流程。

## 4.9.6 小结与练习答案

- 泛化讨论必须明确训练分布与部署分布；
- 协变量偏移改变 $p(x)$，标签偏移改变 $p(y)$，概念偏移改变预测机制；
- 重要性加权要求正确的偏移假设以及训练/测试分布有足够重叠；
- 部署会引入反馈、选择偏差、公平性和因果稳定性问题。

**练习：为什么只监控平均输入值不够？** 不同分布可能拥有相同均值，却在方差、尾部、类别组合和变量依赖关系上完全不同，因此需要多维、分层且与性能关联的监控。